# Notebook 4 (phase 2) — Building a golden set from scratch, small

Notebooks 1 and 3 ended on the same sentence: the public benchmarks cannot tell. HotpotQA is
answered from memory half of the time, QASPER cannot be graded by code, and none of them is
made of documents that look like the ones a real system gets: PDFs with two columns, tables
and footnotes.

This notebook builds the evaluation that fills the gap, and it is deliberately small: a
**corpus** of 200 documents no model has read, and a **golden set** of 50 to 80 questions
on it, each with its kind, its reference answer and the exact sentences that justify it.
The skill shown is not volume: it is building an evaluation from nothing, so that each
question is known to measure something.

Everything is imported from `src/` and run as is. **Live cells** run the whole chain on one
tiny fixture paper (a page of arXiv metadata, a rendered paper, a two-page PDF) with
scripted models: no network, no key, no database. **Archive cells** read what the real run
of 2026-11-05 left in the repository: the manifest of the corpus and the 83 drafts. The
golden set itself does not exist yet: the drafts wait for their review, which is a person's
job (`make golden-review`).

## Part 1 — The problem

Three questions a model could write about a paper, all three useless as evaluation:

| Question | Reference answer | What is wrong |
|---|---|---|
| "Given 30-second windows and a 2-second shift between adjacent frames, how much do consecutive windows overlap?" | 28 seconds | answerable from the question itself: it tests arithmetic, not the system |
| "What method do the authors propose?" | "A two-stage retriever" | which authors? Among 200 papers the question has 200 answers |
| "Which edge hardware platform was the tracker's latency evaluated on?", marked *unanswerable* | (none) | an appendix names it. A system that answers is right and gets punished |

The first and the third were produced by the drafting run of this project; the second is
what the drafting prompt forbids in so many words. A golden set is worth what its worst questions are worth: one wrong
reference answer puts a ceiling on every system, and nobody sees it in an average.

What has to be true of a golden question, and who can check it:

| Property | Checked by |
|---|---|
| the documents are not in the model's training data | a date (part 3.1) |
| the answer is in the document, at a place that can be shown | code: the quoted sentence is looked up, word for word (3.4) |
| the question cannot be answered without the document | a model, asked without the document (3.5) |
| an "unanswerable" question really has no answer anywhere in the document | a second model reading the whole document, then a person (3.5) |
| the question is clear, the reference answer is right, the kind is the right one | **a person, for every question** (3.6) |

The last line is not optional and is not delegated. Everything before it exists to make
that person's two or three hours go to judgment instead of to searching.

A word first: the **kind** of a question. A set made only of "what is X?" says nothing about
the failures that matter. The taxonomy used here has six kinds
([`Category`](../../src/llm_testbench/eval/golden/types.py#L35)), each with the behaviour a good system should show:

| Kind | What it is | A good answer |
|---|---|---|
| factual | one fact, stated in one place | gives the fact |
| multi-hop | needs two passages from different sections | combines both |
| unanswerable | on topic, and the document does not say | refuses |
| ambiguous | two readings, answered differently by the document | names both readings |
| trap | assumes something false about the document | corrects the premise |
| temporal | needs reasoning over dates the document states | reasons over them |

In [1]:
# --- Configuration: edit, then "Run All" ---
import os

from llm_testbench.config import find_repo_root

SAMPLE = os.environ.get("LLM_TESTBENCH_SAMPLE", "0") == "1"
ROOT = find_repo_root()  # the notebook runs from notebooks/phase-2/

# --- Experiment 1 (part 3.1): the selection, on the five fixture candidates -------------
MIN_TABLES = 2  # a paper is kept if its reference text has at least this many tables,
MIN_FOOTNOTES = 2  # this many footnotes,
MIN_DIFFICULTY = 4  # and tables + footnotes + displayed formulas at least this
FRESHNESS_FLOOR = "2026-10-20"  # first version submitted on or after this day

# --- Experiment 2 (part 3.4): which kinds the drafter is asked for ----------------------
KINDS = ["factual", "multi_hop", "unanswerable"]

# --- Experiment 3 (part 3.6): the reviewer's keystrokes, replayed ----------------------
# One decision per draft: a(ccept), r(eject) + a reason, e(dit) + question, answer, kind
# (an empty string keeps what is there).
REVIEW_KEYS = [
    "r",
    "the model answers it without the paper",
    "e",
    "",
    "The dense retriever: 31 ms against 2 ms.",
    "",
    "a",
]

In [2]:
import json
import tempfile
from collections import Counter
from pathlib import Path
from statistics import median

from llm_testbench.config import Settings
from llm_testbench.eval.corpus.arxiv import CC_BY_4, ArxivClient, offline_fetch
from llm_testbench.eval.corpus.build import Corpus, build_corpus, is_candidate
from llm_testbench.eval.corpus.cli import describe
from llm_testbench.eval.corpus.manifest import MANIFEST_PATH, CorpusConfig, Manifest
from llm_testbench.eval.golden.cli import probe_all, status
from llm_testbench.eval.golden.dataset import golden_dataset
from llm_testbench.eval.golden.drafting import Drafter, UnanswerableChecker, locate, paper_text
from llm_testbench.eval.golden.review import review
from llm_testbench.eval.golden.store import (
    CHANGELOG,
    GOLDEN_DIR,
    freeze,
    read_drafts,
    read_questions,
    read_reviews,
    write_drafts,
)
from llm_testbench.eval.golden.types import Category
from llm_testbench.generation.answer import REFUSAL_SENTINEL
from llm_testbench.llm import ModelRef
from llm_testbench.llm.client import LLMClient
from llm_testbench.llm.prompts import PromptRegistry
from llm_testbench.llm.testing import Step, anthropic_message, scripted_anthropic_provider

fixtures = Settings(
    data_dir=ROOT / "data" / "cache", fixtures_dir=ROOT / "tests" / "fixtures", sample=True
)
prompts = PromptRegistry.from_settings(fixtures)
workdir = Path(tempfile.mkdtemp(prefix="golden-notebook-"))  # everything live goes here


def scripted(backend: str, *replies: str) -> LLMClient:
    """A client whose `backend` answers the given texts, in order, then repeats the last:
    the real provider code on a scripted network (no key, no cost)."""
    client = LLMClient.from_settings(fixtures)
    provider, _ = scripted_anthropic_provider(
        [Step(json_body=anthropic_message(reply)) for reply in replies], backend=backend
    )
    client.register_provider(backend, provider)
    return client

## Part 2 — The options

**Where the questions come from.**

| Option | Cost | What you get | Verdict |
|---|---|---|---|
| Written by hand from zero | days for 80 good questions | exactly what you want | too slow, and the taxonomy drifts towards what is easy to write |
| Generated by a model and used as is | minutes, under a dollar | a file of questions | a list of model outputs graded against model outputs: nobody knows which are wrong |
| **Drafted by a model, checked by code, decided by a person** | minutes + two or three hours of review | every question read, the reading made fast | **built here** |

**Which documents.** The brief gives two criteria and two only: published after the models
used, and hard to parse. Three candidates were proposed; the topic was never a criterion.

| Candidate | Formats to break | License | Reference text | Verdict |
|---|---|---|---|---|
| **arXiv papers under CC BY 4.0** | two columns, tables, footnotes, formulas | redistribution with attribution | arXiv renders the LaTeX source to HTML | **chosen** |
| US Federal Register notices | long tables, nested lists, cross-references | public domain | XML, far from the PDF layout | a parser comparison would also compare genres |
| SEC filings (EDGAR) | financial tables, footnotes by the hundred | public | HTML is the original; the PDFs are rare | no PDF problem to measure |

The reference text decided it. To measure what a PDF parser loses (phase 3), and to write
questions whose answers do not depend on a parser, each document needs a text that never
went through one. And arXiv papers are of the same genre as QASPER, so comparing the
in-house numbers with QASPER's does not also compare genres.

Decisions: [ADR 014](../../docs/adr/014-in-house-corpus-and-golden-set.md). What `src/`
holds: `eval/corpus/` (arXiv client, reference text, selection, manifest), `ingest/pdf.py`
(the naive extractor), `eval/golden/` (kinds and states of a question, drafting, review,
frozen set, the set as a dataset of the harness).

## Part 3 — Guided reading

### 3.1 The corpus: fresh, hard to parse, and written down

**Fresh.** [`ArxivClient.harvest`](../../src/llm_testbench/eval/corpus/arxiv.py#L125) asks arXiv's metadata interface (OAI-PMH) for
the records of a period, and [`parse_records`](../../src/llm_testbench/eval/corpus/arxiv.py#L68) keeps, for each, its license and
**the date of its first version**. That detail matters: arXiv lists records by their last
modification, and two days of "recent" records hold papers from 2013 that were revised
yesterday. [`is_candidate`](../../src/llm_testbench/eval/corpus/build.py#L37) keeps a record when it is under the right license,
first submitted on or after the freshness floor, and in a listed category.

The floor itself is not a guess about training data. [`config/corpus.toml`](../../config/corpus.toml)
lists each model used with a day on which it is known to have been **available**; a model
cannot have read what was published after its release. [`CorpusConfig`](../../src/llm_testbench/eval/corpus/manifest.py#L59) refuses a
floor that is not after every model (notebook 3, part 3.2).

**Hard to parse.** [`is_hard_enough`](../../src/llm_testbench/eval/corpus/build.py#L47) counts, on the reference text, what a PDF
extraction gets wrong: tables, footnotes, displayed formulas
([`ReferenceFeatures`](../../src/llm_testbench/eval/corpus/reference.py#L90)). The topic is not looked at.

**Written down.** [`build_corpus`](../../src/llm_testbench/eval/corpus/build.py#L60) examines the candidates in the order of
their identifiers, keeps the first that pass, downloads their PDF, and returns a
[`Manifest`](../../src/llm_testbench/eval/corpus/manifest.py#L133): identifiers, authors, licenses, hashes, features, the floor and
the models it holds for. The manifest is versioned; the documents are not (1.1 GB). Anyone
gets the same documents back with [`download_corpus`](../../src/llm_testbench/eval/corpus/build.py#L220), and
[`Corpus`](../../src/llm_testbench/eval/corpus/build.py#L161) refuses a file that does not match its hash.

The live cell runs the selection on five fixture records, read from files instead of the
network ([`offline_fetch`](../../src/llm_testbench/eval/corpus/arxiv.py#L158)).

In [3]:
selection = CorpusConfig.model_validate(
    {
        "name": "fixture",
        "freshness_floor": FRESHNESS_FLOOR,
        "harvest_until": "2026-11-04",
        "models": {"claude-sonnet-5-5": {"available_by": "2026-10-19", "evidence": "fixture"}},
        "categories": ["cs.CL"],
        "license": CC_BY_4,
        "size": 5,
        "min_tables": MIN_TABLES,
        "min_footnotes": MIN_FOOTNOTES,
        "min_difficulty": MIN_DIFFICULTY,
        "max_characters": 10_000,
    }
)
arxiv = ArxivClient(
    fetch=offline_fetch(ROOT / "tests" / "fixtures" / "arxiv"), sleep=lambda _s: None
)

print(f"{'arXiv id':<12} {'first version':<14} {'category':<9} {'license':<28} candidate")
for record in arxiv.harvest(set_spec="cs", since=selection.freshness_floor):
    license_name = "CC BY 4.0" if record.license == CC_BY_4 else "arXiv non-exclusive (no reuse)"
    print(
        f"{record.arxiv_id:<12} {record.submitted.date()!s:<14} {record.primary_category:<9} "
        f"{license_name:<28} {is_candidate(record, selection)}"
    )

print("\nselection:")
live_manifest = build_corpus(selection, arxiv, workdir / "corpus", echo=print)
entry = live_manifest.documents[0]
print(f"\nkept: {entry.doc_id} — {entry.title!r} by {entry.authors}")
print(f"features: {entry.features}")
print(
    f"fresh for claude-sonnet-5-5: {live_manifest.fresh_for('anthropic/claude-sonnet-5-5')}, "
    f"for a model released later: {live_manifest.fresh_for('anthropic/claude-sonnet-6')}"
)

arXiv id     first version  category  license                      candidate
2610.00001   2026-10-21     cs.CL     CC BY 4.0                    True
1307.02827   2013-07-10     cs.CL     CC BY 4.0                    False
2610.00002   2026-10-22     cs.CL     arXiv non-exclusive (no reuse) False
2610.00003   2026-10-24     cs.CL     CC BY 4.0                    True
2610.00004   2026-10-24     math.CO   CC BY 4.0                    False

selection:
2 fresh candidates under http://creativecommons.org/licenses/by/4.0/
  2610.00001v1: kept (2 tables, 2 footnotes, 1 formulas) — 1/5
  2610.00003v1: no HTML rendering, skipped

kept: 2610.00001v1 — 'A Fresh Paper With Tables' by Ada Lovelace, Charles Babbage
features: sections=4 characters=384 tables=2 footnotes=2 formulas=1 figures=1
fresh for claude-sonnet-5-5: True, for a model released later: False


In [4]:
manifest = Manifest.load(ROOT / MANIFEST_PATH)
print(describe(manifest))
print(f"\nselection rule: {manifest.selection}")

features = [document.features for document in manifest.documents]
print(f"\n{'per document':<22} {'min':>7} {'median':>8} {'max':>8}")
for name in ("characters", "tables", "footnotes", "formulas", "figures"):
    values = [getattr(feature, name) for feature in features]
    print(f"{name:<22} {min(values):>7} {median(values):>8.0f} {max(values):>8}")
with_footnote = sum(1 for feature in features if feature.footnotes)
print(f"\ndocuments with at least one footnote: {with_footnote} of {len(features)}")
print(
    "primary categories:", dict(Counter(d.categories[0] for d in manifest.documents).most_common())
)

arxiv-fresh: 200 documents, first submitted between 2026-10-20 and 2026-10-21 (floor 2026-10-20), 355 candidates seen
median per document: 63,740 characters, 9 tables, 0 footnotes, 11 displayed formulas
fresh for: claude-sonnet-5-5, gpt-5.6-luna, gpt-5.6-terra

selection rule: {'set_spec': 'cs', 'harvest_until': '2026-11-04', 'categories': ['cs.CL', 'cs.IR', 'cs.LG', 'cs.AI', 'cs.CV', 'cs.SE', 'cs.CR', 'cs.DB'], 'license': 'http://creativecommons.org/licenses/by/4.0/', 'size': 200, 'min_difficulty': 12, 'min_tables': 3, 'min_footnotes': 0, 'min_characters': 20000, 'max_characters': 120000, 'order': 'ascending arXiv identifier, first that pass'}

per document               min   median      max
characters               23479    63740   117526
tables                       3        9       33
footnotes                    0        0       14
formulas                     0       11      180
figures                      0        7       47

documents with at least one footnote: 66 of 200
pri

The real corpus: 2,762 fresh candidates under CC BY 4.0 in sixteen days of computer science,
355 examined, **200 kept**, all first submitted on 2026-10-20 or 2026-10-21. The median
document has 9 tables and 11 displayed formulas; a third have a footnote.

### 3.2 Two texts of the same document

A PDF does not contain text in reading order: it contains glyphs at coordinates. Two texts
are built from each document, and they are not meant to agree.

**The reference text**: what a perfect parser would have extracted. arXiv renders the LaTeX
source of a paper to HTML, with sections, paragraphs, tables and footnotes as explicit
elements. [`reference_document`](../../src/llm_testbench/eval/corpus/reference.py#L213) turns that page into sections: a table becomes one
line per row with `|` between cells, a footnote is moved out of its sentence and placed
after the paragraph, a formula is kept as its LaTeX source, the bibliography is dropped.
The golden set is written against this text.

**The naive extraction**: what the simplest code gets out of the PDF.
[`extract_pdf_naive`](../../src/llm_testbench/ingest/pdf.py#L26) asks a library for "the text of the page" and takes what comes:
one section per page, characters in the order they were drawn. It is the baseline phase 3
has to beat, and what the system as built indexes.

The live cell shows both for the fixture paper.

In [5]:
corpus = Corpus(live_manifest, workdir / "corpus")
reference = corpus.reference(entry)
naive = corpus.extracted(entry)

print("===== REFERENCE TEXT (from the HTML rendering) =====")
for section in reference.sections:
    print(f"\n## {section.title}\n{section.text}")
print("\n===== NAIVE EXTRACTION (from the PDF) =====")
for page in naive.sections:
    print(f"\n## {page.title}\n{page.text}")

===== REFERENCE TEXT (from the HTML rendering) =====

## Abstract
We measure two retrievers on a new corpus.

## 1 Introduction
Dense retrieval beats BM25 when k=10.

[footnote] Code at example.org.

The corpus was collected in October 2026.

## 2 Results
Table 1: Recall at 10.

Retriever | Recall
BM25 | 0.61
Dense | 0.74

Table 2: Latency in milliseconds.

BM25 | 2
Dense | 31

## 2.1 Cost
The dense index costs more.

[footnote] Measured on one machine.

c=n\cdot p

Figure 1: Cost against size.

===== NAIVE EXTRACTION (from the PDF) =====

## Page 1
A Fresh Paper With Tables
Abstract
We measure two retrievers on a new corpus.
1 Introduction
Dense retrieval1 beats BM25 when k=10.
The corpus was collected in October 2026.
1Code at example.org.

## Page 2
2 Results
Retriever
Recall
BM25
0.61
Dense
0.74
Table 1: Recall at 10.
BM25
2
Dense
31
Table 2: Latency in milliseconds.
2.1 Cost
The dense index costs more2.
c = n p
2Measured on one machine.
2


Three things the PDF text has lost, all visible above:

- **the table**: its cells come one after the other, with no row and no column. "0.74" is
  no longer the recall of "Dense"; it is a number near a word;
- **the footnotes**: their marks are glued to a word ("retrieval1", "more2"), and their
  text sits wherever the page drew it, here after the last line of the page;
- **the structure**: no sections, only pages; headings are lines like the others, and the
  page number is in the text.

On a real document it looks like this (from the corpus: Shizgal et al., *From DQI to
C-DQI*, arXiv:2610.27937v1, CC BY 4.0). The reference text has a paragraph, then its
footnote:

> `[footnote] We use a 4-point scale. Because DQI assigns each category its own scale, we
> min-max normalize its raw scores to this range: \tilde{y}=1+3\frac{y-y_{\min}}{y_{\max}-y_{\min}} [...]`

The naive extraction of the same page has the footnote in the middle of a sentence of the
body, cut at a hyphen, and the formula as a run of symbols:

> `This final index, representing the overall discourse quality for the transcript, is the
> average over all 1We use a 4-point scale. Because DQI assigns each cat- egory its own
> scale, we min-max normalize its raw scores to this range: ˜y= 1 + 3y−yminymax−ymin [...]`

A chunk cut from that page holds a sentence that never existed. Phase 3 measures what a
real parser recovers; this notebook only needs the two texts to exist.

### 3.3 Three states, three files

A question is never edited in place. It goes through three states
(`eval/golden/types.py`), each in its own file under `data/golden/inhouse/`
(`eval/golden/store.py`):

| State | File | Written by | Used for a measurement? |
|---|---|---|---|
| [`Draft`](../../src/llm_testbench/eval/golden/types.py#L69): what a model proposed, plus what the automatic checks found | `drafts.jsonl` | `make golden-draft`, `make golden-probe` | never |
| [`Review`](../../src/llm_testbench/eval/golden/types.py#L124): a person's decision, accepted, edited or rejected | `reviews.jsonl` (appended; the last decision on a draft wins) | `make golden-review` | never |
| [`GoldenQuestion`](../../src/llm_testbench/eval/golden/types.py#L148): a reviewed question | `questions.jsonl` + `CHANGELOG.md` | `make golden-freeze` | yes, and only this |

### 3.4 Drafting: the model writes, code checks what code can check

[`Drafter.draft`](../../src/llm_testbench/eval/golden/drafting.py#L120) gives one paper to a model with the kinds wanted and gets back
one question per kind: a reference answer, the sentences that justify it, a note. Three
things in it are worth reading.

- **The prompt** ([`prompts/golden_draft.md`](../../prompts/golden_draft.md)) spends half
  its length on one rule: the person asking has not seen the paper. Never "the paper", "the
  authors", "this work"; the question names the method or the dataset it is about.
- **The drafter is not the generator under test.** Questions written by the generator would
  be phrased the way it thinks. Here: `gpt-5.6-terra` drafts, `claude-sonnet-5-5` is
  measured.
- **A quote is evidence only if it is in the paper.** A model asked to quote will sometimes
  paraphrase, or quote a sentence that does not exist. [`locate`](../../src/llm_testbench/eval/golden/drafting.py#L61) looks each
  quote up in the reference text, word for word (whitespace aside), and records the section
  it was found in, or that it was not found. The model's word is never taken.

The drafter reads the beginning of the paper ([`paper_text`](../../src/llm_testbench/eval/golden/drafting.py#L71), the first 24,000
characters in the real run, a third of a median paper): the cost of a draft is mostly the
paper. The kinds are asked in rotation ([`assign_categories`](../../src/llm_testbench/eval/golden/drafting.py#L84)), so that the set
is spread over the taxonomy whatever the papers.

The live cell drafts three questions on the fixture paper. The model is scripted: its
reply is the JSON below, written to contain one honest quote, one paraphrase and one
"unanswerable" claim.

In [6]:
MODEL_REPLY = {
    "questions": [
        {
            "category": "factual",
            "question": "What recall at 10 does the dense retriever reach on the new corpus?",
            "answer": "0.74",
            "quotes": ["Dense | 0.74"],
            "note": "",
        },
        {
            "category": "multi_hop",
            "question": "How much slower is the retriever with the higher recall at 10?",
            "answer": "The dense retriever, 31 ms against 2 ms.",
            "quotes": ["Dense | 0.74", "The dense retriever takes 31 milliseconds."],
            "note": "",
        },
        {
            "category": "unanswerable",
            "question": "On what hardware was the cost of the dense index measured?",
            "answer": "",
            "quotes": [],
            "note": "The text does not name the hardware.",
        },
    ]
}
drafter = Drafter(
    client=scripted("openai", json.dumps(MODEL_REPLY)),
    prompts=prompts,
    model=ModelRef.parse("openai/gpt-5.6-terra"),
    max_characters=400,
)
print("----- what the drafter is given (cut at 400 characters here) -----")
print(paper_text(reference, 400))

drafts = await drafter.draft(reference, [Category(kind) for kind in KINDS])
print("\n----- the drafts, after the quotes were looked up -----")
for draft in drafts:
    print(f"\n[{draft.category.value}] {draft.question}\n  answer: {draft.answer or '(none)'}")
    for quote in draft.quotes:
        where = f"found in {quote.section!r}" if quote.found else "NOT FOUND"
        print(f"  quote ({where}): {quote.text}")
    print(f"  drafter read {draft.characters_read} of {draft.characters_total} characters")
    print(f"  problems: {draft.problems() or 'none seen by code'}")

----- what the drafter is given (cut at 400 characters here) -----
TITLE: A Fresh Paper With Tables

## Abstract
We measure two retrievers on a new corpus.

## 1 Introduction
Dense retrieval beats BM25 when k=10.

[footnote] Code at example.org.

The corpus was collected in October 2026.

## 2 Results
Table 1: Recall at 10.

Retriever | Recall
BM25 | 0.61
Dense | 0.74

Table 2: Latency in milliseconds.

BM25 | 2
Dense | 31

## 2.1 Cost
The dense index costs more.

[footnote] Measured on one machine.

c=n\cdot p

Figure 1: Cost against size.

----- the drafts, after the quotes were looked up -----

[factual] What recall at 10 does the dense retriever reach on the new corpus?
  answer: 0.74
  quote (found in '2 Results'): Dense | 0.74
  drafter read 384 of 384 characters
  problems: none seen by code

[multi_hop] How much slower is the retriever with the higher recall at 10?
  answer: The dense retriever, 31 ms against 2 ms.
  quote (found in '2 Results'): Dense | 0.74
  quote (NOT FOUND

The second quote is a paraphrase: the paper never says "The dense retriever takes 31
milliseconds" (it has a table row, `Dense | 31`). The draft is not thrown away, it is
flagged: the question may be good and the answer right. What code refuses is to present
that sentence to anyone as evidence.

### 3.5 Two more checks before a person looks

Both are models, both are cheap, and neither decides anything.

**Can the question be guessed?** [`probe_all`](../../src/llm_testbench/eval/golden/cli.py#L112) asks each draft to the generator
with no document at all (the closed-book probe of notebook 3). A right answer means the
question does not need the paper: arithmetic on numbers it states, a fact anyone knows, a
yes-or-no with a likely side.

**Is an "unanswerable" draft really unanswerable?** The drafter read a third of the paper.
"The paper does not say" is a claim about all of it. [`UnanswerableChecker.check`](../../src/llm_testbench/eval/golden/drafting.py#L210)
gives the **whole** paper to a second, cheaper model with one job: find the answer and copy
the sentence, or say there is none. Its quote goes through the same word-for-word lookup;
only a passage that is really there reaches the reviewer.

In the live cell the scripted generator answers the first draft from "memory", and the
scripted checker claims that a footnote answers the third.

In [7]:
generator = scripted("anthropic", "It reaches 0.74.", REFUSAL_SENTINEL)
probed = await probe_all(
    drafts,
    client=generator,
    prompts=prompts,
    model=ModelRef.parse("anthropic/claude-sonnet-5-5"),
    ledger=None,
)
checker = UnanswerableChecker(
    client=scripted(
        "openai",
        json.dumps(
            {"answered": True, "answer": "One machine.", "quote": "Measured on one machine."}
        ),
    ),
    prompts=prompts,
    model=ModelRef.parse("openai/gpt-5.6-luna"),
)
checked = []
for draft in probed:
    checked.append(await checker.check(draft, reference))
write_drafts(checked, workdir / "golden")

for draft in checked:
    print(f"[{draft.category.value}] {draft.question}")
    print(
        f"   without any document: {draft.closed_book_answer!r} -> right: {draft.closed_book_correct}"
    )
    if draft.found_elsewhere is not None:
        print(
            f"   second reading found, in {draft.found_elsewhere.section!r}: {draft.found_elsewhere.text!r}"
        )
    print(f"   problems: {draft.problems() or 'none'}\n")

[factual] What recall at 10 does the dense retriever reach on the new corpus?
   without any document: 'It reaches 0.74.' -> right: True
   problems: ['answered correctly without any document: guessable']

[multi_hop] How much slower is the retriever with the higher recall at 10?
   without any document: 'I cannot answer this question from the provided documents.' -> right: False
   problems: ['1 quote(s) not found in the document', 'multi-hop, but the quotes found come from fewer than two sections']

[unanswerable] On what hardware was the cost of the dense index measured?
   without any document: 'I cannot answer this question from the provided documents.' -> right: False
   second reading found, in '2.1 Cost': 'Measured on one machine.'
   problems: ['marked unanswerable, but the paper may answer it (passage below)']



Look at the third draft. The checker found "Measured on one machine." and the lookup
confirmed the sentence is in the paper. But it does not answer "on what hardware": the
second model was wrong, as models are. This is why the check produces a flag and not a
rejection: it tells the reviewer where to look.

### 3.6 The review: one draft, one key

[`review`](../../src/llm_testbench/eval/golden/review.py#L116) shows one draft at a time ([`show`](../../src/llm_testbench/eval/golden/review.py#L60)): its kind and what a
good answer to that kind does, the question, the reference answer, each quote with the
section it was found in, what the two checks said, and the link to the rendered paper. Then
one key: accept, edit, reject. The decision is appended to `reviews.jsonl` at once, so the
session stops anywhere and resumes; a draft reviewed again later overrules the first
decision.

One rule is enforced by the tool: **a question that has an answer cannot be accepted
without a quote found in the document**. Nothing would tell the harness where the answer
is. Such a draft is rejected, or edited into an unanswerable one.

The live cell replays the keystrokes of the configuration cell through the real tool. What
is printed is what the reviewer sees.

In [8]:
keys = iter(REVIEW_KEYS)
recorded = review(
    read_drafts(workdir / "golden"),
    workdir / "golden",
    read=lambda _prompt: next(keys),
    write=print,
)
print(
    f"\ndecisions on file: "
    f"{ {draft_id: item.decision for draft_id, item in read_reviews(workdir / 'golden').items()} }"
)

For each draft, decide whether it can measure a question-answering system.

  Accept when   the question stands on its own (it names what it is about), a person who
                read the document would give the reference answer, and the quotes justify it.
  Edit when     the idea is good and the wording is not: fix the question, the answer or
                the kind. An empty line keeps what is there.
  Reject when   the question can be answered without the document, has several right
                answers that the reference misses, or is not of the kind it claims.

A quote marked NOT FOUND was not copied from the document: the model paraphrased or
invented it. Check the answer against the document itself before accepting.

Keys: a accept, e edit, r reject — s skips, q stops (decisions so far are saved), ? rules.

0 of 3 drafts already reviewed, 3 to go.

──────────────────────────────────────────────────────────────────────────────────────────────────
Draft 2610.00001v1-factual 

### 3.7 Freezing a version

[`freeze`](../../src/llm_testbench/eval/golden/store.py#L67) rebuilds `questions.jsonl` from the drafts and their latest reviews
([`apply_review`](../../src/llm_testbench/eval/golden/types.py#L166)) and writes an entry in the changelog: how many questions of
each kind, how many added, removed and changed since the previous version. A question
keeps its identifier from one version to the next.

The changelog is not bureaucracy. A score on the golden set is a number **about a version
of the set**. Remove five hard questions and every system improves.

In [9]:
questions = freeze(workdir / "golden", version="v1", summary="Fixture paper, three drafts.")
print(f"{len(questions)} questions frozen\n")
for question in questions:
    print(json.dumps(question.model_dump(mode="json", exclude={"quotes"}), indent=1)[:420], "\n")
print((workdir / "golden" / CHANGELOG).read_text())

2 questions frozen

{
 "question_id": "g-2610.00001v1-multi_hop",
 "category": "multi_hop",
 "question": "How much slower is the retriever with the higher recall at 10?",
 "answer": "The dense retriever: 31 ms against 2 ms.",
 "answerable": true,
 "doc_id": "2610.00001v1",
 "note": "",
 "draft_id": "2610.00001v1-multi_hop",
 "edited": true
} 

{
 "question_id": "g-2610.00001v1-unanswerable",
 "category": "unanswerable",
 "question": "On what hardware was the cost of the dense index measured?",
 "answer": "",
 "answerable": false,
 "doc_id": "2610.00001v1",
 "note": "The text does not name the hardware.",
 "draft_id": "2610.00001v1-unanswerable",
 "edited": false
} 

# Changelog of the in-house golden set

Newest first.

## v1 — 2026-11-05

Fixture paper, three drafts.

- 2 questions: 0 factual, 1 multi_hop, 1 unanswerable, 0 ambiguous, 0 trap, 0 temporal.
- Against the previous version: 2 added, 0 removed, 0 changed.
- Rewritten by the reviewer: 1; drafts rejected: 1.



### 3.8 From the golden set to the harness

[`golden_dataset`](../../src/llm_testbench/eval/golden/dataset.py#L65) turns the frozen questions into the question-answering dataset
of notebook 1: a verified quote becomes a gold passage, the reference answer the answer to
find, the kind a label. The corpus to search is the in-house corpus in one of its two
texts, and the choice is the experiment:

- `naive`: what the phase 2 extractor gets out of the PDFs, the system as built;
- `reference`: what the system would score if parsing were solved.

The difference between the two runs is what parsing costs, on the same questions. The live
cell shows why it will not be zero: a quote taken from the reference text may be nowhere in
the naive one.

In [10]:
for text in ("reference", "naive"):
    dataset = golden_dataset(questions, corpus, text=text)
    print(f"{text:<10} {dataset.describe()}")
    document = dataset.corpus[0]
    for example in dataset.examples:
        for passage in example.passages:
            found = locate(passage.text, document).found
            print(f"   [{example.kind}] gold passage {passage.text!r} in the {text} text: {found}")

reference  {'dataset': 'golden/arxiv-fresh', 'split': 'reference', 'examples': 2, 'answerable': 1, 'unanswerable': 1, 'corpus_documents': 1}
   [multi_hop] gold passage 'Dense | 0.74' in the reference text: True
naive      {'dataset': 'golden/arxiv-fresh', 'split': 'naive', 'examples': 2, 'answerable': 1, 'unanswerable': 1, 'corpus_documents': 1}
   [multi_hop] gold passage 'Dense | 0.74' in the naive text: False


`Dense | 0.74` is a row of a table. In the PDF text the two cells are on two lines: the row
does not exist. An end-to-end run on the naive text will be scored against a gold passage
the index cannot contain as written; whether a chunk still holds "at least half of it in
one piece" (the rule of notebook 1) is exactly what the naive arm will measure.

## Part 4 — Run: where the real golden set stands

The archive cell reads the drafts of 2026-11-05 and their checks.

In [11]:
golden = ROOT / GOLDEN_DIR
print(status(golden), "\n")

real = read_drafts(golden)
problems = Counter(problem.split(":")[0] for draft in real for problem in draft.problems())
print("what code found, by kind of problem:")
for problem, count in problems.most_common():
    print(f"  {count:>3}  {problem}")
shares = [d.characters_read / d.characters_total for d in real if d.characters_read]
print(
    f"\nshare of its paper the drafter read: median {median(shares):.0%}, least {min(shares):.0%}"
)

print("\nthe drafts marked unanswerable that the whole paper may answer:")
for draft in real:
    if draft.found_elsewhere is not None:
        print(f"  Q: {draft.question}")
        print(f"     in {draft.found_elsewhere.section!r}: {draft.found_elsewhere.text[:150]}")

frozen = read_questions(golden)
print(
    f"\nfrozen questions: {len(frozen)}"
    + ("" if frozen else " — the set waits for its review (make golden-review)")
)

drafts: 83 — 15 factual, 15 multi_hop, 15 unanswerable, 15 ambiguous, 15 trap, 8 temporal
quotes: 106, of which 0 not found word for word in their document
answered correctly without any document: 2
marked unanswerable although the whole paper may answer: 3
reviewed: 0 — 
frozen set: 0 — 0 factual, 0 multi_hop, 0 unanswerable, 0 ambiguous, 0 trap, 0 temporal 

what code found, by kind of problem:
    3  marked unanswerable, but the paper may answer it (passage below)
    2  answered correctly without any document

share of its paper the drafter read: median 37%, least 17%

the drafts marked unanswerable that the whole paper may answer:
  Q: Which specific pre-trained vision encoder architecture is used by PixelSR for visual semantic guidance?
     in '4.1 Experiment Settings': We use a frozen pre-trained DINOv2-B/14 (Oquab et al., 2023) as the visual semantic encoder E_{\mathrm{vis}}.
  Q: Which six frontier LLMs were benchmarked on BAIT?
     in '6 Experimental Setup': We evaluate six

**83 drafts on 30 papers**, 15 of each kind except temporal (8: asked for a question about
dates, the drafter returned nothing for seven papers, as the prompt allows; research
papers state few dates).

**All 106 quotes were found word for word.** Not one paraphrase. That is the drafter being
good at copying, and it will not hold for every model: the lookup costs nothing and stays.

**The generator, asked without the papers, said it did not know 78 times out of 83** and
gave the reference answer twice, both times by arithmetic on numbers the question states.
Notebook 3 sets that 3% next to the 53% of HotpotQA.

**3 of the 15 drafts marked unanswerable are answered deeper in their paper**: one in five.
The drafter had read 37% of a median paper. The three are printed above with the passage
found; a reviewer would not have reread 60 pages to catch them.

**The golden set does not exist yet.** 83 drafts are not 83 questions. The review is two to
three hours of reading, and what it rejects is the point of doing it.

## Part 5 — Pitfalls

**A first selection rule that kept 2 papers in 70.** I had required two footnotes per
paper, from an idea of what a paper looks like. Counted on the first 24 candidates: tables
are everywhere (median 7), footnotes almost nowhere. The thresholds were set again on that
count, and every page fetched is now kept in the cache, so that trying other thresholds
does not send arXiv the same requests twice ([`build_corpus`](../../src/llm_testbench/eval/corpus/build.py#L60)).

**"Recent" on arXiv means "recently modified".** Two days of records held papers first
submitted in 2013. Freshness is the date of the first version.

**A hash of the page is not a hash of the paper.** arXiv's HTML page carries its own header
and footer, which change without the paper changing. The manifest keeps the fingerprint of
the parsed reference text, not of the page.

**The provider refuses a schema with an optional field.** Structured outputs in strict mode
want every field required and no open object. The first drafting call failed with a 400:
the schema had defaults. A draft "with nothing in it" is now an entry with empty strings,
which the prompt asks for explicitly.

**"Unanswerable" is the kind code can check least, and the kind where a mistake costs
most.** A wrong factual reference lowers every system's score a little. A wrong
unanswerable question punishes exactly the systems that are right. Two models and a person
look at each one.

**A question that cannot stand alone.** "What method do the authors propose?" is a fine
question about one paper and a meaningless one against 200. Unlike QASPER, where the search
is restricted to the question's paper, the golden questions will be searched in the whole
corpus: each must name what it is about. This one only a person checks.

**The reviewer is one person.** There is no measure of how consistent that person is, and
the taxonomy leaves room for it (is this a trap or an ambiguity?). The kinds are labels for
reading results, not a classification anyone should defend to the last question.

**Quoting a paper is redistributing it.** The drafts versioned in a public repository
contain sentences of the papers. That is the reason for the license filter (CC BY 4.0) and
for the manifest naming the authors of every document; a corpus "downloaded by script,
never versioned" would not have been enough.

## Part 6 — Guided experiments

1. **The selection** (part 3.1). Set `MIN_FOOTNOTES = 3`: the fixture paper has two and is
   no longer kept, and the cell stops for lack of a document: that is the first real run,
   with the same rule. Put it back, then set `FRESHNESS_FLOOR = "2026-10-22"`:
   the paper submitted on the 21st stops being a candidate.
2. **The kinds** (part 3.4). Set `KINDS = ["factual", "unanswerable"]`: the scripted model
   still returns three questions, and the drafter keeps the two that were asked for.
   Remove a key from `REVIEW_KEYS` accordingly (two drafts, two decisions).
3. **The review** (part 3.6). Replace the last `"a"` of `REVIEW_KEYS` by `"r", "the
   footnote answers it"`: the frozen set loses its unanswerable question, and the changelog
   says so. Then try to accept the second draft as it is (`"a"` instead of the edit): it has
   a quote that was found, so the tool lets it through; remove that quote from
   `MODEL_REPLY` and the tool refuses.

On the real set: `make golden-status` says where it stands; `make golden-review` opens the
review where it was left.

## Part 7 — The numbers

| | |
|---|---|
| Fresh candidates under CC BY 4.0, computer science, 2026-10-20 to 2026-11-04 | 2,762 |
| Candidates examined / documents kept | 355 / 200 |
| Without an HTML rendering (no reference text) | 16 of the 355 |
| Median document | 63,740 characters, 9 tables, 11 displayed formulas |
| Documents with a footnote | 66 of 200 |
| Drafts / documents drafted on | 83 / 30 |
| Quotes found word for word | 106 of 106 |
| Drafts the generator answers without the paper | 2 of 68 that have an answer |
| "Unanswerable" drafts answered deeper in their paper | 3 of 15 |
| Cost: drafts, closed-book check, second reading | $0.57, $0.07, $0.05 |
| Reviewed questions | 0: the review is next |

There is no published score to compare a golden set with: that is what makes it one. What
it will be compared with is itself: the naive text against the reference text (what parsing
costs), and every technique of phase 3 against this baseline, on a fixed version.

## Part 8 — Interview questions

<details>
<summary>Five questions, with the answers expected (click to open)</summary>

**1. You have no labelled data and two weeks. How do you evaluate a RAG on a client's
documents?**
A small golden set, 50 to 80 questions, built in three steps: a model drafts questions
with a reference answer and the sentences that justify it; code verifies what it can (the
quotes exist, the question is not guessable); a person reviews every question. Spread the
questions over kinds that fail differently: facts, multi-passage, unanswerable, false
premises. Version it. Small and read beats large and unread.

**2. Why not use the model-generated questions directly?**
Because nobody knows which are wrong. Here one "unanswerable" draft in five had its answer
in an appendix the drafter never read. Used as is, the set would have punished the systems
that answer correctly, and the average would have shown nothing.

**3. How do you keep a model from inventing the evidence for its own question?**
Make it quote, and look the quote up in the source text, word for word. A model's claim
that a sentence is in a document is checked by code, never trusted. Same for the second
model that rereads the unanswerable drafts.

**4. Your golden set was written from clean text. The system indexes PDFs. Is that a
problem?**
It is the design. Questions written against a text that never went through a parser can
judge any parser: run the same set on the naive extraction and on the reference text, and
the difference is what parsing costs. Written against the naive extraction, the questions
would only cover what that extraction got right.

**5. A colleague improved the score on the golden set from 61% to 70% by "cleaning it up".
What do you ask?**
Which version of the set each number is on. A score is about a version: removing hard or
ambiguous questions raises every system. The changelog says what was added, removed and
changed; only numbers on the same version are compared.

</details>

## Part 9 — What we did not do, and why

- **The review.** It is Christophe's, by design: two to three hours, `make golden-review`.
  Until then there is no golden set and no number on it.
- **The arms that run it** (naive text, reference text, with the judge of notebook 2 for the
  kinds code cannot grade: trap, ambiguous). They are wired after the first freeze.
- **Questions across several papers.** Every draft is about one paper; a multi-hop question
  combines two sections of it, not two documents. Cross-document questions need a drafter
  that reads several papers at once; to be considered when the set is extended.
- **Scans, forms, spreadsheets.** The corpus is 200 computer-science papers typeset with
  LaTeX. It was chosen for formats that break a PDF extraction and for a reference text to
  measure it against, not for coverage. Phase 3 says so wherever a parser number is quoted.
- **A second reviewer**, and therefore any measure of agreement between reviewers.
- **The whole corpus drafted.** 30 documents of 200: the golden set is small on purpose.
  The other 170 are the haystack the search has to find the answers in, and the reserve for
  the poisoned documents of phase 7.

## Glossary

| Term | Meaning |
|---|---|
| **golden set** | a small set of questions with verified reference answers, written for one corpus |
| **taxonomy** | the kinds a question can be: factual, multi-hop, unanswerable, ambiguous, trap, temporal |
| **draft** | a question as a model proposed it; never used for a measurement |
| **review** | a person's decision on a draft: accepted, edited, rejected |
| **freeze** | writing a version of the reviewed set, with its entry in the changelog |
| **reference answer** | the answer a careful reader of the document would give |
| **quote / evidence** | the sentences of the document that justify the reference answer, copied word for word |
| **guessable** | answerable without the document |
| **manifest** | the versioned list of the corpus: identifiers, authors, licenses, hashes, freshness floor |
| **freshness floor** | the date on or after which every document was first published |
| **reference text** | the document as a perfect parser would have extracted it (here: from arXiv's HTML rendering) |
| **naive extraction** | the text a PDF library returns page by page, with no handling of layout |
| **CC BY 4.0** | a license that allows reuse and redistribution, provided the authors are named |
| **OAI-PMH** | the metadata interface arXiv exposes to list its records by date |